In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

In [2]:
data = pd.read_csv('D:\GenAI\Customer Churn Prediction\Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [12]:
le_gender = LabelEncoder()
data['Gender'] = le_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [13]:
ohe_geo = OneHotEncoder(sparse_output=False)
geography_encoded = ohe_geo.fit_transform(data[['Geography']])
ohe_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [14]:
geo_df = pd.DataFrame(geography_encoded, columns=ohe_geo.get_feature_names_out(['Geography']))
geo_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [11]:
# Combine all the columns into a single DataFrame
final_data = pd.concat([data.drop('Geography', axis=1), geo_df], axis=1)
final_data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [20]:
# Scaling all numerical features
scaler = StandardScaler()
numerical_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
final_data[numerical_features] = scaler.fit_transform(final_data[numerical_features])
final_data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,-0.326221,0,0.293517,-1.041760,-1.225848,-0.911583,1,1,0.021886,1,1.0,0.0,0.0
1,-0.440036,0,0.198164,-1.387538,0.117350,-0.911583,0,1,0.216534,0,0.0,0.0,1.0
2,-1.536794,0,0.293517,1.032908,1.333053,2.527057,1,0,0.240687,1,1.0,0.0,0.0
3,0.501521,0,0.007457,-1.387538,-1.225848,0.807737,0,0,-0.108918,0,1.0,0.0,0.0
4,2.063884,0,0.388871,-1.041760,0.785728,-0.911583,1,1,-0.365276,0,0.0,0.0,1.0


In [21]:
# Save the encoders and scaler for future use
import pickle
with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(le_gender, f)
with open('onehot_encoder_geo.pkl', 'wb') as f:
    pickle.dump(ohe_geo, f)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [22]:
# Divide the data into features and target variable
X = final_data.drop('Exited', axis=1)
y = final_data['Exited']

In [23]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ANN Implementation

In [25]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [26]:
# build our ANN model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

d:\GenAI\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [27]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
# compile the model
model.compile(optimizer='adam', 
loss='binary_crossentropy', 
metrics=['accuracy', 'Precision', 'Recall']
)

In [29]:
# Set up the tensorboard
log_dir = "Logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [30]:
# Set up earlystopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [31]:
# Train the model
history = model.fit(
    X_train, y_train, 
    validation_split=0.2, 
    epochs=100, 
    batch_size=32, 
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - Precision: 0.6556 - Recall: 0.1339 - accuracy: 0.8066 - loss: 0.4446 - val_Precision: 0.8614 - val_Recall: 0.2702 - val_accuracy: 0.8444 - val_loss: 0.3872
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - Precision: 0.7455 - Recall: 0.3744 - accuracy: 0.8444 - loss: 0.3759 - val_Precision: 0.8168 - val_Recall: 0.3323 - val_accuracy: 0.8506 - val_loss: 0.3594
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - Precision: 0.7345 - Recall: 0.4478 - accuracy: 0.8525 - loss: 0.3538 - val_Precision: 0.8244 - val_Recall: 0.3354 - val_accuracy: 0.8519 - val_loss: 0.3569
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - Precision: 0.7339 - Recall: 0.4569 - accuracy: 0.8536 - loss: 0.3472 - val_Precision: 0.7121 - val_Recall: 0.4379 - val_accuracy: 0.8512 - val_loss: 0.3449
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - Precision: 0.7477 - Recall: 0.4841 - accuracy: 0.8597 - loss: 0.3406 - val_Precision: 0.6515 - 

In [32]:
model.save('model.h5')

In [33]:
# load tensorboard Extension
%load_ext tensorboard

In [35]:
%tensorboard --logdir 'D:\GenAI\Customer Churn Prediction\Logs\fit' --port=6006

ERROR: Failed to launch TensorBoard (exited with 4294967295).
Contents of stderr:
I0000 00:00:1775800965.305964    7852 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775800977.591628    7852 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
E0410 11:33:16.600550  7852 program.py:300] TensorBoard could not bind to port 6006, it was already in use
ERROR: TensorBoard could not bind to port 6006, it was already in use

In [36]:
history_df = pd.DataFrame(history.history)
history_df.to_csv('model_history.csv', index=False)
history_df.head()

,Precision,Recall,accuracy,loss,val_Precision,val_Recall,val_accuracy,val_loss
0,0.655556,0.133888,0.806562,0.444583,0.861386,0.270186,0.844375,0.387239
1,0.745482,0.374433,0.844375,0.375858,0.816794,0.332298,0.850625,0.359380
2,0.734491,0.447806,0.852500,0.353769,0.824427,0.335404,0.851875,0.356905
3,0.733900,0.456884,0.853594,0.347189,0.712121,0.437888,0.851250,0.344881
4,0.747664,0.484115,0.859688,0.340581,0.651452,0.487578,0.844375,0.348240


In [39]:
# Load the trained models and pickle files
from tensorflow.keras.models import load_model

model = load_model('model.h5')

with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)
with open('onehot_encoder_geo.pkl', 'rb') as f:
    onehot_encoder_geo = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [40]:
# Example input data for prediction
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}


In [43]:
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

d:\GenAI\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [47]:
input_df = pd.DataFrame([input_data])
input_df = input_df.drop('Geography', axis=1)
input_df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,Male,40,3,60000,2,1,1,50000


In [50]:
print(label_encoder_gender.classes_)
input_df = pd.DataFrame([input_data])
print(input_df['Gender'])

[0 1]
0    Male
Name: Gender, dtype: str


In [52]:
# Re-fit the encoder on the actual words
label_encoder_gender.fit(['Male', 'Female'])

# Now this will work and return [1] or [0]
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [61]:
input_df = pd.concat([input_df, geo_encoded_df], axis=1)
input_df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [59]:
input_df.drop(['Geography_France', 'Geography_Germany', 'Geography_Spain'], axis=1, inplace=True)

In [65]:
# scale numeric features
scaled_numeric = scaler.transform(input_df[numerical_features])

# keep the other columns as they are
other_cols = input_df[['Gender', 'HasCrCard', 'IsActiveMember',
                       'Geography_France', 'Geography_Germany', 'Geography_Spain']].to_numpy()

# combine them in the same order used during training
full_input = np.concatenate([scaled_numeric, other_cols], axis=1)



In [67]:
prediction = model.predict(full_input)
churn_prob = prediction[0][0]
print("Churn Probability:", churn_prob)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
Churn Probability: 1.0


In [68]:
if churn_prob > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is unlikely to churn.")

The customer is likely to churn.
